In [86]:
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import pandas as pd
import numpy  as np
import math

pd.set_option('display.max_columns', None)

In [87]:
REF_VAR = 'phi'
RAW_VAR = 'wx'
TARGET_TIME = (100, 400) 
VIEW_LIMITS = (0.5, 0.7)

# DADOS

In [88]:
df = pd.read_csv('files/pitch_cal.csv')
df

,az,wx,wz,yaw,ay,time,ax,e,roll,wy,pitch,phi,phi_d,phi_dd,cal_a_w,cal_a_h,cal_w,cal_o,cal_heave,heave
0,0.010817,0.05242,-0.23137,83.617,-9.838904,5.072000,0.152513,0.000000,-94.338,-0.13189,-89.244,0.000000,0.000000e+00,0.000000e+00,0.000000,-9.806650,0.000000e+00,0.000000,0.000000,0.000000
1,0.033411,-0.28030,0.00031,83.489,-9.812024,5.082000,0.187895,0.000000,-94.209,0.05542,-89.244,0.000000,0.000000e+00,0.000000e+00,0.000000,-9.806650,0.000000e+00,0.000000,0.000000,0.000000
2,-0.000049,0.26531,-0.17762,83.545,-9.722685,5.092000,0.151268,0.000000,-94.265,0.56119,-89.245,0.000000,0.000000e+00,0.000000e+00,0.000000,-9.806650,0.000000e+00,0.000000,0.000000,0.000000
3,-0.082415,-0.09454,-0.31389,83.423,-9.756930,5.102000,0.083131,0.000000,-94.144,-0.40448,-89.244,0.000000,0.000000e+00,0.000000e+00,0.000000,-9.806650,0.000000e+00,0.000000,0.000000,0.000000
4,0.058791,-0.12867,0.22484,83.327,-9.750144,5.112000,0.150856,0.000000,-94.048,0.05397,-89.245,0.000000,0.000000e+00,0.000000e+00,0.000000,-9.806650,0.000000e+00,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35745,0.068647,0.25778,0.04314,45.947,-9.804993,435.851990,0.133165,0.299999,-58.744,0.12483,-89.069,0.299999,-6.078984e-13,-1.751295e-12,0.051347,-9.806516,-6.078984e-13,0.299999,0.000027,0.000027
35746,0.020104,0.21597,-0.10998,45.860,-9.784850,435.862000,0.047141,0.299999,-58.657,-0.02844,-89.069,0.299999,-6.254793e-13,-1.764900e-12,0.051347,-9.806516,-6.254793e-13,0.299999,0.000027,0.000027
35747,0.092045,-0.33553,-0.29463,45.899,-9.760363,435.872009,0.202851,0.299999,-58.695,0.03189,-89.070,0.299999,-6.431964e-13,-1.778505e-12,0.051347,-9.806516,-6.431964e-13,0.299999,0.000027,0.000027
35748,0.095683,0.13819,-0.58705,45.839,-9.826018,435.881989,0.131242,0.299999,-58.635,0.33036,-89.069,0.299999,-6.610494e-13,-1.792110e-12,0.051347,-9.806516,-6.610494e-13,0.299999,0.000027,0.000027


In [89]:
plt.figure(figsize=(12, 5))
plt.plot(df.time, df[RAW_VAR], label=f'Valor Alvo ({RAW_VAR})')
plt.plot(df.time, df[REF_VAR], label=f'Referência ({REF_VAR})')
plt.grid(alpha=.3); plt.legend(); plt.xlabel('time'); plt.ylabel('response')
plt.show()

/tmp/ipykernel_47688/1759514893.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [90]:
df = df.loc[(df.time >= TARGET_TIME[0]) & (df.time <= TARGET_TIME[1])]
df.loc[:, 'time'] = df.time.values - df.time.values[0]

plt.figure(figsize=(12, 5))
plt.plot(df.time, df[RAW_VAR], label=f'Valor Alvo ({RAW_VAR})')
plt.plot(df.time, df[REF_VAR], label=f'Referência ({REF_VAR})')
plt.grid(alpha=.3); plt.legend(); plt.xlabel('time'); plt.ylabel('response')
plt.show()

/tmp/ipykernel_47688/262705251.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [91]:
VIEW_TIME = (df.time.values[-1]*VIEW_LIMITS[0], df.time.values[-1]*VIEW_LIMITS[1])
target    = df.loc[(df.time >= VIEW_TIME[0]) & (df.time <= VIEW_TIME[1])]

plt.figure(figsize=(12, 5))
plt.plot(target.time, target[RAW_VAR], label=f'Valor Alvo ({RAW_VAR})')
plt.plot(target.time, target[REF_VAR], label=f'Referência ({REF_VAR})')
plt.grid(alpha=.3); plt.legend(); plt.xlabel('time'); plt.ylabel('response')
plt.show()

/tmp/ipykernel_47688/541605608.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# DEFASAGEM

In [92]:
class Phaser:
    def __init__(self, target, reference):
        self.target    = target
        self.reference = reference

    def get(self, df):
        x_norm = df[self.target]    - df[self.target].mean()
        y_norm = df[self.reference] - df[self.reference].mean()
        
        correlation = np.correlate(y_norm, x_norm, mode='full')
        lags = np.arange(-len(df) + 1, len(df))
        lag  = lags[np.argmax(np.abs(correlation))]
        return lag

    def set(self, df, lag):
        if lag == 0:
            return df

        df.loc[:, self.target] = df[self.target].shift(lag)
        df = df.dropna().reset_index(drop=True)
        return df

target = df.loc[(df.time >= VIEW_TIME[0]) & (df.time <= VIEW_TIME[1])]
phaser = Phaser(RAW_VAR, REF_VAR)
lag    = phaser.get(target)
print(lag, 'samples')

51 samples


In [93]:
df = phaser.set(df, lag)
df

,az,wx,wz,yaw,ay,time,ax,e,roll,wy,pitch,phi,phi_d,phi_dd,cal_a_w,cal_a_h,cal_w,cal_o,cal_heave,heave
0,-0.170136,32.17279,1.49334,312.468,-9.545568,0.639999,0.249050,17.400003,35.230,-0.23019,-73.554,7.692066,-1.503084,-17.839066,0.689899,-9.715113,-1.503084,7.692066,0.091519,0.091519
1,-0.170136,32.17279,1.49334,312.468,-9.545568,0.639999,0.249050,17.400003,35.230,-0.23019,-73.554,7.662903,-1.999939,-17.781766,0.686944,-9.716856,-1.999939,7.662903,0.091519,0.091519
2,-0.202488,31.47926,1.07661,312.464,-9.569800,0.650002,0.365543,17.400003,35.234,-0.59104,-73.530,7.624440,-2.495161,-17.707742,0.682993,-9.719107,-2.495161,7.624440,0.091519,0.091519
3,-0.188680,31.47926,-0.58589,312.473,-9.593924,0.669998,0.367357,17.400003,35.225,-0.01822,-73.510,7.581696,-2.988298,-17.617074,0.678894,-9.721746,-2.988298,7.581696,0.091519,0.091519
4,-0.180187,31.07961,-0.83005,312.474,-9.498162,0.680000,0.341821,17.400003,35.223,-0.06641,-73.516,7.533072,-3.478901,-17.509745,0.674376,-9.724802,-3.478901,7.533072,0.091519,0.091519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24314,0.046523,0.08582,0.15148,44.818,-9.783016,299.849998,0.046238,0.299999,-57.445,0.13210,-89.072,0.299999,0.004272,0.013610,0.051822,-9.806519,0.004272,0.299999,0.000027,0.000027
24315,0.054966,0.07310,-0.18120,44.934,-9.777956,299.869987,0.091859,0.299999,-57.561,-0.10607,-89.072,0.299999,0.004331,0.013231,0.051809,-9.806519,0.004331,0.299999,0.000027,0.000027
24316,0.054966,0.27363,-0.18120,44.934,-9.777956,299.869987,0.091859,0.299999,-57.561,-0.10607,-89.072,0.299999,0.004369,0.012832,0.051795,-9.806519,0.004369,0.299999,0.000027,0.000027
24317,0.073432,0.02877,0.12044,44.861,-9.778623,299.879997,0.147394,0.299999,-57.488,-0.51374,-89.071,0.299999,0.004386,0.012416,0.051781,-9.806519,0.004386,0.299999,0.000027,0.000027


In [94]:
VIEW_TIME = (df.time.values[-1]*VIEW_LIMITS[0], df.time.values[-1]*VIEW_LIMITS[1])
target    = df.loc[(df.time >= VIEW_TIME[0]) & (df.time <= VIEW_TIME[1])]

plt.figure(figsize=(12, 5))
plt.plot(target.time, target[RAW_VAR], label=f'Valor Alvo ({RAW_VAR})')
plt.plot(target.time, target[REF_VAR], label=f'Referência ({REF_VAR})')
plt.grid(alpha=.3); plt.legend(); plt.xlabel('time'); plt.ylabel('response')
plt.show()

/tmp/ipykernel_47688/541605608.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# AMOSTRAGEM

In [95]:
def gaussian(data):
    data  = np.array(data)
    n     = data.shape[0]
    mu    = data.mean()
    sigma = data.std()

    x  = np.linspace(mu - 4*sigma, mu + 4*sigma, 400)
    y  = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x - mu)/sigma)**2)
    plt.title(f'Distribuição Gaussiana')
    plt.hist(data, density=True, alpha=0.75)
    plt.plot(x, y)
    
    text = f'média:     {mu:.3f}±{sigma:.3f} \nmediana: {np.median(data):.3f}'
    opts = dict(boxstyle='round', facecolor='white', alpha=0.8)
    plt.text(0.05, 0.95, text, transform=plt.gca().transAxes, verticalalignment='top', bbox=opts)
    plt.xlabel('x'); plt.ylabel('frequência'); plt.grid()

target = df.loc[(df.time >= VIEW_TIME[0]) & (df.time <= VIEW_TIME[1])]
time = df.time.diff()[1:].to_numpy()
dt   = np.median(time).round(3)
print(dt)
gaussian(time)

0.01


In [96]:
def normalizePeriod(df, key, dt=0.15):
    df = df.copy().sort_values(key)
    df[key] = df[key] - df[key].iloc[0]

    initTime  = df[key].iloc[0]
    finalTime = df[key].iloc[-1] + dt
    n = int(np.floor((finalTime - initTime) / dt)) + 1
    newAxis = np.round(np.linspace(initTime, initTime + dt*(n-1), n), 10)
    target  = pd.DataFrame({key: newAxis})
    out = pd.merge_asof(target, df, on=key, direction='backward')
    return out


df = normalizePeriod(df, 'time', dt)
df

,time,az,wx,wz,yaw,ay,ax,e,roll,wy,pitch,phi,phi_d,phi_dd,cal_a_w,cal_a_h,cal_w,cal_o,cal_heave,heave
0,0.00,-0.170136,32.17279,1.49334,312.468,-9.545568,0.249050,17.400003,35.230,-0.23019,-73.554,7.662903,-1.999939,-17.781766,0.686944,-9.716856,-1.999939,7.662903,0.091519,0.091519
1,0.01,-0.170136,32.17279,1.49334,312.468,-9.545568,0.249050,17.400003,35.230,-0.23019,-73.554,7.662903,-1.999939,-17.781766,0.686944,-9.716856,-1.999939,7.662903,0.091519,0.091519
2,0.02,-0.202488,31.47926,1.07661,312.464,-9.569800,0.365543,17.400003,35.234,-0.59104,-73.530,7.624440,-2.495161,-17.707742,0.682993,-9.719107,-2.495161,7.624440,0.091519,0.091519
3,0.03,-0.188680,31.47926,-0.58589,312.473,-9.593924,0.367357,17.400003,35.225,-0.01822,-73.510,7.581696,-2.988298,-17.617074,0.678894,-9.721746,-2.988298,7.581696,0.091519,0.091519
4,0.04,-0.188680,31.47926,-0.58589,312.473,-9.593924,0.367357,17.400003,35.225,-0.01822,-73.510,7.581696,-2.988298,-17.617074,0.678894,-9.721746,-2.988298,7.581696,0.091519,0.091519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29922,299.22,0.046523,0.08582,0.15148,44.818,-9.783016,0.046238,0.299999,-57.445,0.13210,-89.072,0.299999,0.004272,0.013610,0.051822,-9.806519,0.004272,0.299999,0.000027,0.000027
29923,299.23,0.054966,0.27363,-0.18120,44.934,-9.777956,0.091859,0.299999,-57.561,-0.10607,-89.072,0.299999,0.004369,0.012832,0.051795,-9.806519,0.004369,0.299999,0.000027,0.000027
29924,299.24,0.073432,0.02877,0.12044,44.861,-9.778623,0.147394,0.299999,-57.488,-0.51374,-89.071,0.299999,0.004386,0.012416,0.051781,-9.806519,0.004386,0.299999,0.000027,0.000027
29925,299.25,0.073432,0.02877,0.12044,44.861,-9.778623,0.147394,0.299999,-57.488,-0.51374,-89.071,0.299999,0.004386,0.012416,0.051781,-9.806519,0.004386,0.299999,0.000027,0.000027


# CURVE FIT

In [97]:
from scipy.optimize import curve_fit, OptimizeWarning
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=OptimizeWarning)

In [98]:
class TemporalFit:
    def __init__(self, time, xData, yData, max_iter=2100000000):
        self.time  = np.array(time)
        self.xData = np.array(xData)
        self.yData = np.array(yData)
        self.max_iter = max_iter
        self.coefs = []

    def f(self, x, a, b):
        return a*x + b

    def apply(self, data):
        if len(self.coefs) == 0:
            return None

        return self.f(np.array(data), *self.coefs)
    
    def update(self):
        result      = curve_fit(self.f, self.xData, self.yData, maxfev=self.max_iter)
        self.coefs  = [float(round(coef, 12)) for coef in list(result[0])]
        self.yModel = self.apply(self.xData)
        
        self.error = (self.yData - self.yModel)
        self.rmse  = np.sqrt(np.mean(self.error**2))
        self.mae   = np.mean(np.abs(self.error))
        self.max_error = np.max(np.abs(self.error))

        self.scale_factor_error = np.abs(1.0 - self.coefs[0]) * 100 
        self.std_noise = self.error.std()
        
        ss_res = np.sum(self.error**2) 
        ss_tot = np.sum((self.yData - np.mean(self.yData))**2) 

        if ss_tot == 0:
            self.r2 = 1.0 if ss_res == 0 else 0.0
        else:
            self.r2 = 1 - float(ss_res / ss_tot)

    def plot(self):
        plt.figure(figsize=(20, 10))
        plt.subplot(2, 2, 1)
        plt.plot(self.time, self.xData)
        plt.xlabel('time'); plt.ylabel('response')
        plt.grid(); plt.title('Raw Value')
        
        plt.subplot(2, 2, 2)
        plt.plot(self.time, self.yData)
        plt.xlabel('time'); plt.ylabel('response')
        plt.grid(); plt.title('Reference Value')

        plt.subplot(2, 2, 3)
        plt.plot(self.time, self.yData, color='blue', label='reference')
        plt.plot(self.time, self.yModel, color='red', label='model')
        plt.xlabel('time'); plt.ylabel('response')
        plt.legend(); plt.title(f'R2 Score {self.r2:.3f}')
        plt.grid()

        plt.subplot(2, 2, 4)
        plt.plot(self.time, self.error, color='blue')
        plt.xlabel('time'); plt.ylabel('response')
        plt.ylim(-self.max_error*3, self.max_error*3)
        plt.title(f'temporal error (max={self.max_error:.3f} | rmse={self.rmse:.3f})')
        plt.grid(); plt.show()

    def display(self):
        display(pd.DataFrame([{'r2': self.r2, 'mae': self.mae, 'rmse': self.rmse, 'max_error': self.max_error}]))

In [99]:
target = df.loc[(df.time >= VIEW_TIME[0]) & (df.time <= VIEW_TIME[1])]
model = TemporalFit(target.time, target[RAW_VAR], target[REF_VAR])
model.update()
model.display()

erro_medio = model.mae
erro_pico  = model.rmse
escala     = model.coefs[0] # Sensibilidade (o 'a' de ax + b)
bias       = model.coefs[1] # Offset (o 'b' de ax + b)
precision  = 2 * model.error.std() # Calculando o intervalo de 95% de confiança

print(f'Análise de Calibração - {RAW_VAR}')
print(f'Precisão: ±{precision:.4f} m/s² (95% de confiança)')
print(f'Bias do Sensor:  {bias:.4f} m/s²')
print(f'Fator de Escala: {escala:.4f}')
model.plot()

,r2,mae,rmse,max_error
0,0.980012,0.187131,0.232315,1.003095


Análise de Calibração - wx
Precisão: ±0.4646 m/s² (95% de confiança)
Bias do Sensor:  0.1966 m/s²
Fator de Escala: 0.2462


/tmp/ipykernel_47688/523156151.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(); plt.show()


In [100]:
df['model'] = model.apply(df[RAW_VAR])

plt.figure(figsize=(12, 5))
plt.plot(df.time, df.model, label=f'Valor Alvo ({RAW_VAR})')
plt.plot(df.time, df[REF_VAR], label=f'Referência ({REF_VAR})')
plt.grid(alpha=.3); plt.legend(); plt.xlabel('time'); plt.ylabel('response')
plt.show()

/tmp/ipykernel_47688/2604037051.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [101]:
import pandas as pd

class Calibration:
    def __init__(self, model_fit, variable_name):
        self.model    = model_fit
        self.var_name = variable_name
        self.max_noise_std       = 0.0
        self.max_scale_error_pct = 0.0
        
    def evaluate(self, value, threshold):
        return "Passed" if value <= threshold else "Failed"

    def get(self):
        df = pd.DataFrame({
            'Test': [
                f'STD sensor noise [{self.var_name}]', 
                f'RMS scale factor error [%]'
            ],
            'Test requirement': [
                self.max_noise_std, 
                self.max_scale_error_pct
            ],
            'Value Measured': [
                round(self.model.std_noise, 4),
                round(self.model.scale_factor_error, 4)
            ],
            'Status': [
                self.evaluate(self.model.std_noise, self.max_noise_std),
                self.evaluate(self.model.scale_factor_error, self.max_scale_error_pct)
            ]
        })
        
        df.set_index('Test', inplace=True)
        return df



calibration = Calibration(model, RAW_VAR)
calibration.max_noise_std       = 0.05
calibration.max_scale_error_pct = 0.10
calibration.get()

,Test requirement,Value Measured,Status
Test,,,
STD sensor noise [wx],0.05,0.2323,Failed
RMS scale factor error [%],0.10,75.3804,Failed


# GERANDO RELATÓRIO

In [102]:
import os
import io
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PURPLE_MID  = '#6A3CBC'
PINK_ACCENT = '#E91E63'

class ResultExporter:
    def __init__(self, basePath, rawVar, model, calibration=None):
        self.basePath = basePath
        self.rawVar = rawVar
        self.model = model
        self.calibration = calibration
        self.outputDir = os.path.join(basePath, 'release', rawVar)

    def _ensureDir(self):
        os.makedirs(self.outputDir, exist_ok=True)

    def _makePlot(self, plotType, width=250, height=150):
        fig, ax = plt.subplots(figsize=(width / 72, height / 72), dpi=150)
        model = self.model
        label = self.rawVar.capitalize()

        if plotType == 'ref':
            ax.plot(model.time, model.yData, color=PURPLE_MID, linewidth=0.8, label='Referência')
            ax.plot(model.time, model.yModel, color=PINK_ACCENT, linewidth=0.8, label='Medido')
            ax.legend(fontsize=6, loc='upper right')
            ax.set_ylabel('Ângulo (deg)', fontsize=7)
            ax.set_title(f'{label}: Referência vs. Medido', fontsize=8, fontweight='bold')
        else:
            ax.plot(model.time, model.error, color=PURPLE_MID, linewidth=0.8, label='Erro')
            ax.legend(fontsize=6, loc='upper right')
            ax.set_ylabel('Erro (deg)', fontsize=7)
            ax.set_ylim(-max(model.error*3.5), max(model.error*3.5))
            ax.set_title(f'Erro Temporal – {label}', fontsize=8, fontweight='bold')

        ax.set_xlabel('Tempo (s)', fontsize=7)
        ax.tick_params(axis='both', labelsize=6)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        plt.tight_layout(pad=0.5)
        return fig

    def exportPlots(self):
        self._ensureDir()

        figRef = self._makePlot('ref')
        refPath = os.path.join(self.outputDir, 'ref_vs_model.png')
        figRef.savefig(refPath, format='png', dpi=150, bbox_inches='tight')
        plt.close(figRef)

        figErr = self._makePlot('error')
        errPath = os.path.join(self.outputDir, 'error.png')
        figErr.savefig(errPath, format='png', dpi=150, bbox_inches='tight')
        plt.close(figErr)

        return refPath, errPath

    def exportMetrics(self):
        self._ensureDir()
        model = self.model

        precision = 2 * model.error.std()

        metrics = {
            'r2': float(round(model.r2, 6)),
            'mae': float(round(model.mae, 6)),
            'rmse': float(round(model.rmse, 6)),
            'maxError': float(round(model.max_error, 6)),
            'scaleFactorError': float(round(model.scale_factor_error, 6)),
            'stdNoise': float(round(model.std_noise, 6)),
            'coefs': [float(c) for c in model.coefs],
            'precision': float(round(precision, 6)),
        }

        path = os.path.join(self.outputDir, 'metrics.json')
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(metrics, f, indent=2, ensure_ascii=False)

        return path

    def exportCalibration(self):
        self._ensureDir()
        if self.calibration is None:
            return None

        calDf = self.calibration.get()
        tests = []
        for idx, row in calDf.iterrows():
            tests.append({
                'name': str(idx),
                'requirement': float(row['Test requirement']),
                'measured': float(row['Value Measured']),
                'status': str(row['Status']),
            })

        data = {'tests': tests}
        path = os.path.join(self.outputDir, 'calibration.json')
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        return path

    def export(self):
        self._ensureDir()
        refPath, errPath = self.exportPlots()
        metricsPath = self.exportMetrics()
        calPath = self.exportCalibration()

        print(f'[export] Resultados exportados para: {self.outputDir}')
        print(f'  → {refPath}')
        print(f'  → {errPath}')
        print(f'  → {metricsPath}')
        if calPath:
            print(f'  → {calPath}')

        return self.outputDir


RAW_VAR = 'roll'
exporter = ResultExporter('.', RAW_VAR, model)
exporter.export()

[export] Resultados exportados para: ./release/roll
  → ./release/roll/ref_vs_model.png
  → ./release/roll/error.png
  → ./release/roll/metrics.json


'./release/roll'